In [1]:
import torch
import torch.nn as nn
from deviceConfig import device

# To insta relaod modules in Jupyter Notebook
%load_ext autoreload
%autoreload 2

In [9]:
with open("dataset/combined_corpus.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [10]:
len(text)

27226679

In [13]:
# loading the tokenized data

raw_data = torch.load("trained_tokenizer/train.bin")

In [14]:
raw_data.shape

torch.Size([10216206])

In [15]:
raw_data[:10]

tensor([ 70, 800, 403, 889, 122, 263,  58,  10,  66, 101])

In [16]:
# We dont need to use a dataloader as the tokenized data is already in a 1D tensor format. We can directly use it for training.

n = int(0.8 * len(raw_data))  # first 80% will be train, rest val

train_data = raw_data[:n]
val_data = raw_data[n:]


In [17]:
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size
        

    def __len__(self):
        # The total number of valid starting positions
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # Slice a context window of length 'block_size'
        x = self.data[idx : idx + self.block_size]
        # Slice the exact same window, shifted one token to the right
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y


train_dataset = GPTDataset(train_data, block_size=256)
val_dataset = GPTDataset(val_data, block_size=256)


In [18]:
from torch.utils.data import DataLoader
import os


Batch_size = 256

train_dataloader = DataLoader(train_dataset, 
                              batch_size=Batch_size, 
                              shuffle=True,
                              num_workers=0
)

val_dataloader = DataLoader(val_dataset,
                            batch_size=Batch_size,
                            shuffle=False,
                            num_workers=0
)

In [20]:
from model_code.moe_gpt import MoEGPT

model_0 = MoEGPT()

state_dict= torch.load("models/MoE_model",weights_only=True)

model_0.load_state_dict(state_dict)

model_0.to(device=device)


MoEGPT(
  (token_embedding_table): Embedding(1500, 384)
  (position_embedding_table): Embedding(256, 384)
  (blocks): ModuleList(
    (0-3): 4 x Block_MoE(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-3): 4 x Head(
            (key): Linear(in_features=384, out_features=96, bias=False)
            (value): Linear(in_features=384, out_features=96, bias=False)
            (query): Linear(in_features=384, out_features=96, bias=False)
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (ffwd): sparseMoEFeedForward(
        (router_1): Router(
          (router): Linear(in_features=384, out_features=4, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
        (experts): ModuleList(
          (0-3): 4 x expert(
            (net): Sequential(
              (0): Linear(in_features=384, out_

In [21]:
optimizer = torch.optim.AdamW(model_0.parameters(), 
                              lr=3e-4)

In [22]:
next(model_0.parameters()).device

device(type='mps', index=0)

In [23]:
from loops_for_model.training_loops import train_loop

train_losses, val_losses = train_loop(
    model=model_0, 
    train_data_loader=train_dataloader, 
    validation_data_loader=val_dataloader,
    optimizer=optimizer, 
    epochs=1, 
    device=device
)


/Users/hartej/Hartej46/Merismos/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
  0%|          | 0/31925 [00:00<?, ?it/s]

  0%|          | 1/31925 [01:17<690:17:48, 77.84s/it]


KeyboardInterrupt: 

In [7]:
import plotly.express as px
import pandas as pd

# Combine the lists into a DataFrame for Plotly
loss_df = pd.DataFrame({
    'Epoch': range(1, len(train_losses) + 1),
    'Training Loss': train_losses,
    'Validation Loss': val_losses
})

fig = px.line(
    loss_df, 
    x='Epoch', 
    y=['Training Loss', 'Validation Loss'], 
    title='MoEGPT Training vs. Validation Loss',
    markers=True
)

fig.update_layout(
    yaxis_title="Loss", 
    xaxis_title="Epoch",
    legend_title_text=None,
    hovermode="x unified"
)

fig.show()

ModuleNotFoundError: No module named 'plotly'